In [1]:
import subprocess, sys, os, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode

sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate torchao bitsandbytes pycocoevalcap 'trl==0.29.1' 2>&1 | tail -3")
sh(f'{sys.executable} -c "import trl, transformers, peft, torch; print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.cuda.get_device_name(0))"')
sh(f'{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.



0.29.1 5.20.2 0.21.2 2.11.0+cu128 Tesla T4
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so



  Successfully uninstalled torchao-0.18.0



0

In [2]:
import os, glob, json, shutil, hashlib
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()

# dataset script: chỉ nhận thư mục có đủ 4 tệp (loại grpo_spice.py nằm trong output train)
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/gen_test_grpo.py", recursive=True)
       if all(os.path.exists(os.path.join(os.path.dirname(p), f))
              for f in ("grpo_spice.py", "build_branch_data.py", "preds_s1_seed101.jsonl"))]
assert len(SRC) == 1, f"DỪNG: cần đúng một grpo-spice-script bản mới, thấy {SRC}"
SRC = SRC[0]
for f in ("grpo_spice.py", "build_branch_data.py", "gen_test_grpo.py"):
    shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
assert md5(f"{W}/grpo_spice.py") == "07ea87b6d156d1faa391a4274dffd7cf", "DỪNG: grpo_spice.py bản cũ"
assert md5(f"{W}/gen_test_grpo.py") == "b18ec1aa3fd037ddffb9411db63c2a3f", "DỪNG: gen_test_grpo.py lệch máy nhà"
S1P = f"{SRC}/preds_s1_seed101.jsonl"
assert sum(1 for _ in open(S1P)) == 6958

BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d)
MERGED = f"{W}/s1_merged"

def la_grpo(d):
    c = os.path.join(d, "adapter_config.json")
    return os.path.exists(c) and "s1_merged" in json.load(open(c)).get("base_model_name_or_path", "")
CK = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/checkpoint-500/adapter_model.safetensors", recursive=True)
      if la_grpo(os.path.dirname(f)) and "/ref" not in f]
assert len(CK) == 1, f"DỪNG: cần đúng một checkpoint-500 của xao101, thấy {CK}"
CK500 = CK[0]
assert "xao101" in CK500, f"DỪNG: adapter không từ output xao101-2: {CK500}"
assert json.load(open(f"{CK500}/trainer_state.json"))["global_step"] == 500, "DỪNG: không phải bước 500"
print("md5 adapter xao101/500:", md5(f"{CK500}/adapter_model.safetensors"), flush=True)

# thesis-score: gói mã cũ (score_run đã chấm S1 59,11) + test_ac đủ ảnh
best = None
for t in glob.glob("/kaggle/input/**/test_ac/test.jsonl", recursive=True):
    root = os.path.dirname(t)
    nimg = len(glob.glob(f"{root}/images/*.png"))
    if sum(1 for _ in open(t)) == 6958 and nimg >= 4463 and os.path.exists(f"{root}/ocr.jsonl"):
        best = root
assert best, "DỪNG: không thấy test_ac đủ 6.958 dòng + 4.463 ảnh"
TA = best
assert md5(f"{TA}/test.jsonl") == "da58299ee551a926a21cbecb5232bf78", "DỪNG: test.jsonl lệch máy nhà"
PKG = os.path.dirname(os.path.dirname(os.path.dirname(TA)))          # …/thesis/harness/dg1_cache/test_ac → …/thesis
shutil.copytree(f"{PKG}/harness", f"{W}/thesis/harness", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("images"))
link = f"{W}/thesis/harness/dg1_cache/test_ac/images"
if os.path.lexists(link): os.remove(link)
os.symlink(f"{TA}/images", link)
WS = f"{W}/thesis"

print("SRC   ", SRC); print("BUNDLE", BUNDLE); print("CK500 ", CK500); print("TA    ", TA)
print("có thư mục ref/ trong ck500:", os.path.isdir(f"{CK500}/ref"), flush=True)


md5 adapter xao101/500: d6ff6487195d96a6b25c2d35ad88f6d9


SRC    /kaggle/input/datasets/uyenle0712/grpo-spice-script
BUNDLE /kaggle/input/datasets/uyenle0712/fgrb-p1-bundle/fgrb_p1_bundle
CK500  /kaggle/input/notebooks/uyenle0712/xao101-2/grpo_spice/checkpoint-500
TA     /kaggle/input/datasets/uyenle0712/thesis-score/thesis/harness/dg1_cache/test_ac
có thư mục ref/ trong ck500: True


In [3]:
TEST = False

def chay(cmd, log, cwd, nhip=120, tran_gio=None):
    t0 = time.time()
    with open(log, "w") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT,
                             env={**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                                  "PYTHONUNBUFFERED": "1"})
        while q.poll() is None:
            time.sleep(nhip if not TEST else 30)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"  {(time.time()-t0)/3600:5.2f} h · {os.path.basename(log)} · {L[-1][:110] if L else 'chưa có dòng nào'}", flush=True)
            if tran_gio and time.time() - t0 > tran_gio * 3600:
                q.terminate(); print(f"⛔ cắt ở {tran_gio} h để còn giờ chấm", flush=True); break
    print(f"--- {os.path.basename(log)} · mã thoát {q.returncode} · {(time.time()-t0)/3600:.2f} h ---")
    print("".join(open(log, errors="ignore").readlines()[-6:]), flush=True)
    return q.returncode

r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED],
                   cwd=W, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0, "DỪNG: hoà lỗi"


[hoà] xong → /kaggle/working/s1_merged · dtype torch.float16



In [4]:
TA_ARGS = ["--recs", f"{TA}/test.jsonl", "--ocr", f"{TA}/ocr.jsonl", "--images", f"{TA}/images", "--tap-only", "--no-q4"]
out = f"{W}/kiem_hoa_test.jsonl"
if os.path.exists(out): os.remove(out)
chay(["python", "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--s1", S1P,
      "--n", "20", "--out", out] + TA_ARGS, f"{W}/kiem_hoa_test.log", W)
dong = [l for l in open(f"{W}/kiem_hoa_test.log") if "[kiểm hoà]" in l]
print(dong)
x = int(dong[-1].split("]")[1].split("/")[0])
assert x >= 16, f"DỪNG: kiểm hoà {x}/20 < 16 — đừng gắn điểm lưu, gửi log về"


   0.03 h · kiem_hoa_test.log · [kiểm hoà] 16/20 trùng preds_s1_seed101


--- kiem_hoa_test.log · mã thoát 0 · 0.03 h ---
[điểm lưu] KHÔNG gắn — đây là S1 hoà
[nối tiếp] đã có 0/20
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[xong] 20 câu · rỗng 0
[kiểm hoà] 16/20 trùng preds_s1_seed101



['[kiểm hoà] 16/20 trùng preds_s1_seed101\n']


In [5]:
PRED = f"{W}/pred_ck500_xao101_test{'_thu' if TEST else ''}.jsonl"
chay(["python", "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--ckpt", CK500,
      "--out", PRED] + (["--n", "5"] if TEST else []) + TA_ARGS,
     f"{W}/gen_test{'_thu' if TEST else ''}.log", W, tran_gio=6)
n = sum(1 for _ in open(PRED))
print(f"xao101 test: {n}/{5 if TEST else 4463} câu", flush=True)
assert n == (5 if TEST else 4463), "DỪNG: thiếu câu — chạy lại ô này (nối tiếp được), không xoá tệp"


   0.03 h · gen_test.log · warnings.warn(


   0.07 h · gen_test.log · 50/4463 · 2.4 phút · 2.88 s/bước · 'Click on the plus icon of the List item.'


   0.10 h · gen_test.log · 100/4463 · 4.9 phút · 2.92 s/bước · 'Remove the current form name'


   0.13 h · gen_test.log · 150/4463 · 7.3 phút · 2.91 s/bước · 'Click on the first shoes option.'


   0.17 h · gen_test.log · 150/4463 · 7.3 phút · 2.91 s/bước · 'Click on the first shoes option.'


...

   3.60 h · gen_test.log · 4350/4463 · 213.5 phút · 2.94 s/bước · 'click on View Product Details'


   3.63 h · gen_test.log · 4400/4463 · 215.9 phút · 2.94 s/bước · 'Type My Daily Thought in name bar'


   3.67 h · gen_test.log · [xong] 4463 câu · rỗng 1


--- gen_test.log · mã thoát 0 · 3.67 h ---
  4250/4463 · 208.6 phút · 2.95 s/bước · 'Click on the send icon at the top of the screen.'
  4300/4463 · 211.1 phút · 2.95 s/bước · 'Click on the year tab.'
  4350/4463 · 213.5 phút · 2.94 s/bước · 'click on View Product Details'
  4400/4463 · 215.9 phút · 2.94 s/bước · 'Type My Daily Thought in name bar'
  4450/4463 · 218.4 phút · 2.94 s/bước · 'Click on the Add label option'
[xong] 4463 câu · rỗng 1



xao101 test: 4463/4463 câu


In [6]:
SC = f"{W}/score_ck500_xao101_test{'_thu' if TEST else ''}.json"
cmd = ["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground",
       "--preds", PRED, "--out", SC] + (["--n", "3"] if TEST else [])
chay(cmd, f"{W}/cham_test{'_thu' if TEST else ''}.log", WS)
R = [json.loads(l) for l in open(SC.replace(".json", "_raw.jsonl"))]
print(f"== xao101 test: {len(R)} bước · exec {sum(int(r.get('executable', 0)) for r in R)}", flush=True)
print("KIỂM (bản đủ): phải 4463 bước. Đừng tự diễn giải — gửi về máy nhà.")


   0.03 h · cham_test.log · Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate l


   0.07 h · cham_test.log · [14:00:09] 40/4463 =  0.9% · 0.23 bước/giây · còn ~327 phút


   0.10 h · cham_test.log · [14:01:34] 60/4463 =  1.3% · 0.23 bước/giây · còn ~322 phút


   0.13 h · cham_test.log · [14:04:25] 100/4463 =  2.2% · 0.23 bước/giây · còn ~315 phút


   0.17 h · cham_test.log · [14:05:51] 120/4463 =  2.7% · 0.23 bước/giây · còn ~313 phút


...

   5.30 h · cham_test.log · [19:14:16] 4440/4463 = 99.5% · 0.23 bước/giây · còn ~2 phút


   5.33 h · cham_test.log · /kaggle/working/score_ck500_xao101_test_raw.jsonl  (4463 bước — đổi luật chấm hay thêm lát cắt thì chấm lại từ


--- cham_test.log · mã thoát 0 · 5.33 h ---
  ô-Voronoi tâm (headline):  59.1%   KTC95 [57.3%, 60.8%]
  đĩa dung sai (báo kèm) :  69.3%   KTC95 [67.7%, 70.9%]
  cụm: 1091 (hiệu dụng 454.3)

Đã lưu /kaggle/working/score_ck500_xao101_test.json
         /kaggle/working/score_ck500_xao101_test_raw.jsonl  (4463 bước — đổi luật chấm hay thêm lát cắt thì chấm lại từ tệp này, KHÔNG gọi lại bộ trỏ)



== xao101 test: 4463 bước · exec 2638


KIỂM (bản đủ): phải 4463 bước. Đừng tự diễn giải — gửi về máy nhà.


In [ ]:
import zipfile, glob
fs = [f for f in glob.glob("/kaggle/working/*") if os.path.isfile(f) and not f.endswith(".zip") and "s1_merged" not in f
      and os.path.getsize(f) < 200*2**20]
with zipfile.ZipFile("/kaggle/working/ck500_xao101_test.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in fs: z.write(f, os.path.basename(f))
print("đã zip", len(fs), "tệp:", sorted(os.path.basename(f) for f in fs), flush=True)
